# Assignment 2 — AgeDB Dataset Preparation

Downloads AgeDB from Hugging Face and creates a 200-identity subset with at least 2 images per identity. Images at age 5 or above are preferred, but age 5 is not a hard cutoff. For each identity, the pair with the largest available age gap is selected.


In [ ]:
!pip -q install datasets pandas pillow matplotlib


In [ ]:
from datasets import load_dataset

DATASET_NAME = "marcelohaps/agedb"
dataset = load_dataset(DATASET_NAME)
print(dataset)


In [ ]:
print("Splits:", dataset.keys())
split_name = list(dataset.keys())[0]
data = dataset[split_name]
print("Using split:", split_name)
print("Number of images:", len(data))
print("Columns:", data.column_names)
print("First record:")
print(data[0])


In [ ]:
import pandas as pd

cols = data.column_names
identity_col = next((c for c in ["identity","person_id","subject_id","id"] if c in cols), None)
age_col = next((c for c in ["age","Age"] if c in cols), None)
gender_col = next((c for c in ["gender","Gender","sex","Sex"] if c in cols), None)
print("Identity:", identity_col, "Age:", age_col, "Gender:", gender_col)
if identity_col is None or age_col is None:
    raise ValueError("Identity/age column not found. Inspect data.column_names above.")

rows=[]
for i in range(len(data)):
    r=data[i]
    rows.append({"dataset_index":i,"identity":r[identity_col],"age":r[age_col],"gender":r[gender_col] if gender_col else "unknown"})
meta=pd.DataFrame(rows)
meta["age"]=pd.to_numeric(meta["age"],errors="coerce")
meta=meta.dropna(subset=["identity","age"]).reset_index(drop=True)
print(meta.head())
print("Usable records:",len(meta))


In [ ]:
counts=meta.groupby("identity").size()
valid=counts[counts>=2]
print("Total identities:",meta.identity.nunique())
print("Identities with >=2 images:",len(valid))
if len(valid)<200:
    raise ValueError("Fewer than 200 identities have at least 2 usable images.")


In [ ]:
PREFERRED_MIN_AGE=5

def best_pair(group):
    preferred=group[group.age>=PREFERRED_MIN_AGE]
    candidates=preferred if len(preferred)>=2 else group
    if len(candidates)<2: return None
    candidates=candidates.sort_values("age")
    a,b=candidates.iloc[0],candidates.iloc[-1]
    return {"identity":a.identity,"index1":int(a.dataset_index),"index2":int(b.dataset_index),"age1":float(a.age),"age2":float(b.age),"age_gap":abs(float(b.age)-float(a.age)),"gender":a.gender}

pairs=[p for _,g in meta.groupby("identity") if (p:=best_pair(g))]
pairs_df=pd.DataFrame(pairs).sort_values("age_gap",ascending=False).reset_index(drop=True)
print("Usable identity pairs:",len(pairs_df))
print(pairs_df.head(10))


In [ ]:
NUM_SUBJECTS=200
selected=pairs_df.head(NUM_SUBJECTS).copy()
selected.insert(0,"subject_id",[f"subject_{i:03d}" for i in range(1,len(selected)+1)])
print("Subjects:",len(selected))
print("Images:",len(selected)*2)
print("Mean age gap:",round(selected.age_gap.mean(),2))
print("Age gap range:",selected.age_gap.min(),"to",selected.age_gap.max())
print("\nGender distribution:")
print(selected.gender.value_counts(dropna=False))


In [ ]:
from pathlib import Path
from PIL import Image
import io

OUTPUT_DIR=Path("/content/AgeDB_Assignment2")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
final_rows=[]

for _,p in selected.iterrows():
    subject_dir=OUTPUT_DIR/p.subject_id
    subject_dir.mkdir(exist_ok=True)
    ordered=sorted([(int(p.index1),p.age1),(int(p.index2),p.age2)],key=lambda x:x[1])
    for n,(idx,age) in enumerate(ordered,1):
        obj=data[idx]["image"]
        if isinstance(obj,Image.Image): img=obj.convert("RGB")
        elif isinstance(obj,dict) and "bytes" in obj: img=Image.open(io.BytesIO(obj["bytes"])).convert("RGB")
        else: raise TypeError(f"Unsupported image type: {type(obj)}")
        name=f"{p.subject_id}_img{n}_age{int(age)}.jpg"
        out=subject_dir/name
        img.save(out,quality=95)
        final_rows.append({"subject_id":p.subject_id,"identity":p.identity,"image_number":n,"age":age,"age_gap":p.age_gap,"gender":p.gender,"dataset_index":idx,"path":str(out)})

final_meta=pd.DataFrame(final_rows)
final_meta.to_csv(OUTPUT_DIR/"metadata.csv",index=False)
print("Created:",OUTPUT_DIR)
print("Subjects:",final_meta.subject_id.nunique())
print("Images:",len(final_meta))


In [ ]:
subject_counts=final_meta.groupby("subject_id").size()
print("Minimum images/subject:",subject_counts.min())
print("Minimum age:",final_meta.age.min())
print("Maximum age:",final_meta.age.max())
print("Average age:",round(final_meta.age.mean(),2))
print("Average age gap:",round(final_meta.groupby("subject_id").age_gap.first().mean(),2))
assert len(subject_counts)>=200
assert subject_counts.min()>=2
print("\nBasic assignment requirements passed.")


In [ ]:
import matplotlib.pyplot as plt
age_gaps=selected.age_gap
plt.figure(figsize=(8,5))
plt.hist(age_gaps,bins=20)
plt.xlabel("Age gap (years)")
plt.ylabel("Number of subjects")
plt.title("Age Gap Distribution — Selected AgeDB Subset")
plt.show()


In [ ]:
import shutil
zip_path=shutil.make_archive("/content/AgeDB_Assignment2","zip",OUTPUT_DIR)
print("ZIP:",zip_path)


In [ ]:
from google.colab import files
files.download("/content/AgeDB_Assignment2.zip")


## Next stage

Use this generated subset for MTCNN detection, preprocessing, FaceNet/PFE embeddings, genuine/impostor pair generation, Gaussian noise, salt-and-pepper noise, synthetic/GAN images, and FMR/FNMR/ROC/EER analysis.
